## Setup

In [54]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from tigramite import data_processing as pp
from tigramite import plotting as tp
from tigramite.independence_tests.cmiknn import CMIknn
from tigramite.independence_tests.cmisymb import CMIsymb
from tigramite.independence_tests.gsquared import Gsquared
from tigramite.independence_tests.parcorr import ParCorr
from tigramite.lpcmci import LPCMCI

from csi_vae_gumbel.settings import Settings

settings = Settings()

IND_TEST = "cmisymb"  # "parcorr", "gsquared", "cmisymb", "cmiknn"

## Dataset

We use the complete dataset for causal inference, withouy any train/test split.

In [ ]:
# Shape will be (T, latent_dim, n_categories)
latents = np.load(f"../{settings.study_path}/latents/latents_hard.npy")

# Shape will be (T,)
labels = np.load(f"../{settings.study_path}/latents/labels.npy")

match IND_TEST:
    case "parcorr" | "cmiknn":
        # Get the probability of the most likely category for each latent dimension
        latents = torch.softmax(torch.from_numpy(latents), dim=2).numpy()
        latents = np.max(latents, axis=2)
    case "gsquared" | "cmisymb":
        # Get the argmax indices (the most likely category) for each latent dimension
        latents = np.argmax(latents, axis=2)

labelled_causal_data = {}

# 2. Split and THEN shift per activity
for label in np.unique(labels):
    # Extract latents for this specific movement sequence
    activity_latents = latents[labels == label]
    activity_latents = activity_latents[::15]
    labelled_causal_data[int(label)] = activity_latents

## Causal Analysis

In [ ]:
# 1. Setup the analysis parameters
match IND_TEST:
    case "parcorr":
        ind_test = ParCorr(significance="analytic")
    case "gsquared":
        ind_test = Gsquared()
    case "cmisymb":
        ind_test = CMIsymb()
    case "cmiknn":
        ind_test = CMIknn()

# Correct names: We have 'latent_dim' variables, each taking 'n_categories' values
var_names = [f"Z{i}" for i in range(latents.shape[1])]

tau_min = 1  # minimum time lag (e.g., t-1)
tau_max = 2  # maximum time lag (e.g., t-2)

# 2. Iterate through each activity
for label_id, data in labelled_causal_data.items():
    activity_name = settings.activities[label_id]

    # Initialize Tigramite DataFrame
    # Note: 'datatypes' should be 'discrete' for Gsquared
    match IND_TEST:
        case "parcorr" | "cmiknn":
            dataframe = pp.DataFrame(data, var_names=var_names)
        case "gsquared" | "cmisymb":
            dataframe = pp.DataFrame(data, var_names=var_names, data_type=np.ones_like(data))

    # 3. Run LPCMCI
    # LPCMCI is robust to latent common causes (unobserved drivers in CSI)
    lpcmci = LPCMCI(dataframe=dataframe, cond_ind_test=ind_test, verbosity=0)

    # pc_alpha is the significance level for the conditional independence tests
    results = lpcmci.run_lpcmci(
        tau_min=tau_min,
        tau_max=tau_max,
        pc_alpha=0.01,
    )

    # 4. Plotting
    # Time series graph is best for t, t-1, t-2 visualization
    fig, ax = tp.plot_time_series_graph(
        figsize=(4, 4),
        val_matrix=results["val_matrix"],
        graph=results["graph"],
        var_names=var_names,
        link_colorbar_label="MCI Strength",
    )

    plt.title(f"Temporal Latent Causal Graph: {activity_name}")
    plt.show()

/mnt/servicesdata/lcotti/csi-vae-gumbel/.venv/lib/python3.13/site-packages/tigramite/independence_tests/cmisymb.py:218: NumbaWarning: 
Compilation is falling back to object mode WITHOUT looplifting enabled because Function "parallelize_shuffles" failed type inference due to: - Resolution failure for literal arguments:
No implementation of function Function(<function np_all at 0x7fa61e05ccc0>) found for signature:

 >>> np_all(array(bool, 2d, C), axis=Literal[int](1))

There are 2 candidate implementations:
  - Of which 2 did not match due to:
  Overload in function 'np_all': File: numba/np/arraymath.py: Line 801.
    With argument(s): '(array(bool, 2d, C), axis=int64)':
   Rejected as the implementation raised a specific error:
     TypingError: got an unexpected keyword argument 'axis'
  raised from /mnt/servicesdata/lcotti/csi-vae-gumbel/.venv/lib/python3.13/site-packages/numba/core/typing/templates.py:791

- Resolution failure for non-literal arguments:
No implementation of function